# AURA VTO  -  Google Colab GPU Inference Server (Tesla T4)

This notebook prepares and executes the self-hosted **FastAPI VTO Service** (`services/vto_gpu/app.py`)
running the decoupled **MMDiT + DWPose** neural pipeline on a Tesla T4 GPU in Google Colab.

### Key Architecture & Guarantees
- **Automated Dependency Installation**: Installs all required packages directly from `services/vto/requirements.txt`.
- **Preserves Colab CUDA Stack**: Does not corrupt or overwrite Colab's pre-installed CUDA/PyTorch runtime.
- **Direct Weights Download**: Automatically downloads official, verified model weights directly into `/content/aura/services/vto/weights/` from official Hugging Face repositories.
- **Strict SHA-256 Verification**: Every file is cryptographically validated against `docs/vto/VTO_ARTIFACT_MANIFEST.json`. Downloads abort immediately upon any mismatch.
- **Idempotent**: Re-running skips files that are already present and verified.
- **Zero Commercial Hosted APIs**: Pure self-hosted open-source pipeline (Apache-2.0).
- **Secure ngrok Tunnel**: Exposes port 8001 via HTTPS for the AURA mobile app (`EXPO_PUBLIC_VTO_COLAB_URL`).

### Prerequisites
1. Switch runtime to GPU: **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU**.
2. Add your secrets in the Colab sidebar (**Key icon - Secrets**):
   - `SUPABASE_URL`: e.g. `https://your-project.supabase.co`
   - `SUPABASE_SERVICE_ROLE_KEY`: Your service role key
   - `NGROK_AUTHTOKEN`: Free auth token from [dashboard.ngrok.com](https://dashboard.ngrok.com)
   - `VTO_JWT_SECRET`: (Optional if using Supabase ES256 JWKS verification)
   - `VTO_WEIGHTS_DIR`: (Optional: defaults automatically to `/content/aura/services/vto/weights`)

In [ ]:
# 1. Verify Tesla T4 GPU Allocation
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: No GPU detected! Please go to Runtime -> Change runtime type and select T4 GPU.")


In [ ]:
# 2. Clean Environment Setup & Dependency Installation
import os
import sys
import subprocess
from pathlib import Path

# 1. Locate or clone genuine AURA repository
AURA_ROOT = Path("/content/aura")
if not (AURA_ROOT / "services" / "vto_gpu" / "app.py").exists():
    if Path("services/vto_gpu/app.py").exists():
        AURA_ROOT = Path.cwd()
        print(f"Detected AURA repository at current directory: {AURA_ROOT}")
    elif Path("/content/auramain/services/vto_gpu/app.py").exists():
        AURA_ROOT = Path("/content/auramain")
        print(f"Detected AURA repository at: {AURA_ROOT}")
    else:
        print("Cloning genuine AURA repository into /content/aura...")
        !git clone https://github.com/Shriyash05/auramain.git /content/aura
        AURA_ROOT = Path("/content/aura")

# Change working directory to repository root
%cd {str(AURA_ROOT)}

# 2. Create required directory structure
(AURA_ROOT / "services" / "vto" / "weights" / "dwpose").mkdir(parents=True, exist_ok=True)

# 3. Install strictly pinned AURA VTO dependencies from services/vto/requirements.txt
req_path = AURA_ROOT / "services" / "vto" / "requirements.txt"
if not req_path.exists():
    req_path = AURA_ROOT / "services" / "vto_gpu" / "requirements.txt"

print(f"Installing dependencies from {req_path}...")
!pip -q install -r {str(req_path)}
print("[OK] Requirements installed successfully.")

# 4. Immediate in-process dependency & hardware verification
print("\n--- Validating Installed Runtime & Import Paths ---")
required_modules = [
    ("PyTorch", "torch"),
    ("TorchVision", "torchvision"),
    ("ONNX Runtime GPU", "onnxruntime"),
    ("OpenCV (Headless)", "cv2"),
    ("NumPy", "numpy"),
    ("Pillow", "PIL"),
    ("Matplotlib", "matplotlib"),
    ("Einops", "einops"),
    ("SafeTensors", "safetensors"),
    ("HuggingFace Hub", "huggingface_hub"),
    ("FastAPI", "fastapi"),
    ("Uvicorn", "uvicorn"),
    ("Python-Jose", "jose"),
    ("Supabase SDK", "supabase"),
    ("PyNgrok", "pyngrok"),
]

import_failures = []
for label, mod_name in required_modules:
    try:
        mod = __import__(mod_name)
        ver = getattr(mod, "__version__", "available")
        print(f"  [OK] {label:20s} : OK (version: {ver})")
    except Exception as err:
        print(f"  [FAIL] {label:20s} : FAIL -> {err}")
        import_failures.append(f"{label} ({mod_name}): {err}")

if import_failures:
    raise SystemExit("BLOCKED: AURA VTO core dependency import failure(s):\n  " + "\n  ".join(import_failures))

import onnxruntime as ort
providers = ort.get_available_providers()
print(f"\nAvailable ONNX Execution Providers: {providers}")
if "CUDAExecutionProvider" not in providers:
    print("WARNING: CUDAExecutionProvider not active in ONNX Runtime. Running DWPose on CPU fallback.")
else:
    print("[OK] ONNX CUDAExecutionProvider: VERIFIED AVAILABLE for DWPose on Tesla T4.")


In [ ]:
# 3. Load Secrets & Server Configuration
import os
try:
    from google.colab import userdata
    for secret_key in ['SUPABASE_URL', 'SUPABASE_SERVICE_ROLE_KEY', 'VTO_JWT_SECRET', 'VTO_WEIGHTS_DIR', 'NGROK_AUTHTOKEN']:
        try:
            val = userdata.get(secret_key)
            if val:
                os.environ[secret_key] = val
        except Exception:
            pass
except ImportError:
    pass

# Check required secrets
required = ['SUPABASE_URL', 'SUPABASE_SERVICE_ROLE_KEY']
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise SystemExit(f"BLOCKED: Missing required Colab Secrets: {', '.join(missing)}\nPlease add them using the Key icon in the Colab sidebar.")

# Set standard AURA VTO defaults
os.environ.setdefault('VTO_WEIGHTS_DIR', str(AURA_ROOT / 'services' / 'vto' / 'weights'))
os.environ.setdefault('VTO_MODEL_RESOLUTION', '672,432')
os.environ.setdefault('VTO_INPUT_BUCKET', 'vto_inputs')
os.environ.setdefault('VTO_OUTPUT_BUCKET', 'vto_results')

print("[OK] Secrets and environment variables configured:")
print(f"  - SUPABASE_URL:          {os.environ['SUPABASE_URL']}")
print(f"  - VTO_WEIGHTS_DIR:       {os.environ['VTO_WEIGHTS_DIR']}")
print(f"  - VTO_MODEL_RESOLUTION:  {os.environ['VTO_MODEL_RESOLUTION']}")
print(f"  - VTO_INPUT_BUCKET:      {os.environ['VTO_INPUT_BUCKET']}")
print(f"  - VTO_OUTPUT_BUCKET:     {os.environ['VTO_OUTPUT_BUCKET']}")


In [ ]:
# 4. Direct Model-Weights Download & Strict SHA-256 Verification
import hashlib
import json
import os
import sys
import time
import urllib.request
from pathlib import Path

weights_dir = Path(os.environ['VTO_WEIGHTS_DIR'])
dwpose_dir = weights_dir / "dwpose"
weights_dir.mkdir(parents=True, exist_ok=True)
dwpose_dir.mkdir(parents=True, exist_ok=True)

# Exact official sources and SHA-256 hashes from docs/vto/VTO_ARTIFACT_MANIFEST.json
ARTIFACTS = [
    {
        "name": "model.safetensors",
        "dest": weights_dir / "model.safetensors",
        "url": "https://huggingface.co/fashn-ai/fashn-vton-1.5/resolve/7720683168567eb5a2a4c67f15116c6e29c83ded/model.safetensors",
        "sha256": "d6cd38286885bc29fa487ea9383f80ffeb95862e7747c630d42c5d3c05bdd35a",
        "expected_bytes": 1943668048,  # ~1.81 GiB / 1.94 GB
        "description": "FASHN VTON v1.5 MMDiT Diffusion Checkpoint"
    },
    {
        "name": "dwpose/yolox_l.onnx",
        "dest": dwpose_dir / "yolox_l.onnx",
        "url": "https://huggingface.co/fashn-ai/DWPose/resolve/548b5df25b84d9f4aac0611dfa1c2a7a12f15571/yolox_l.onnx",
        "sha256": "7860ae79de6c89a3c1eb72ae9a2756c0ccfbe04b7791bb5880afabd97855a411",
        "expected_bytes": 216746733,   # ~206.7 MiB / 216.7 MB
        "description": "DWPose YOLOX-L Person Detector ONNX"
    },
    {
        "name": "dwpose/dw-ll_ucoco_384.onnx",
        "dest": dwpose_dir / "dw-ll_ucoco_384.onnx",
        "url": "https://huggingface.co/fashn-ai/DWPose/resolve/548b5df25b84d9f4aac0611dfa1c2a7a12f15571/dw-ll_ucoco_384.onnx",
        "sha256": "724f4ff2439ed61afb86fb8a1951ec39c6220682803b4a8bd4f598cd913b1843",
        "expected_bytes": 134399116,   # ~128.2 MiB / 134.4 MB
        "description": "DWPose U-COCO Whole-Body Keypoint Estimator ONNX"
    }
]

total_expected_bytes = sum(a["expected_bytes"] for a in ARTIFACTS)
print("=" * 70)
print("* AURA VTO MODEL WEIGHTS DOWNLOAD & VERIFICATION")
print("=" * 70)
print(f"Target Directory:     {weights_dir}")
print(f"Total Expected Size:  {total_expected_bytes / (1024**3):.2f} GiB ({total_expected_bytes / (10**9):.2f} GB)")
print("Artifacts to verify:")
for idx, a in enumerate(ARTIFACTS, 1):
    print(f"  {idx}. {a['name']:<28} ({a['expected_bytes'] / (1024**2):.1f} MiB) -> {a['description']}")
print("=" * 70)

def compute_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(8 * 1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

def download_file(url: str, dest: Path, expected_bytes: int):
    part_dest = Path(str(dest) + ".download_part")
    if part_dest.exists():
        part_dest.unlink()

    headers = {"User-Agent": "AURA-VTO-Colab-Downloader/1.0"}
    req = urllib.request.Request(url, headers=headers)

    start_time = time.time()
    downloaded = 0
    last_print = 0

    with urllib.request.urlopen(req) as resp, open(part_dest, "wb") as out:
        content_length = int(resp.headers.get("content-length", expected_bytes))
        block_size = 8 * 1024 * 1024  # 8 MiB chunks for high-speed transfer

        while True:
            chunk = resp.read(block_size)
            if not chunk:
                break
            out.write(chunk)
            downloaded += len(chunk)

            # Update progress once every 0.5s
            now = time.time()
            if now - last_print > 0.5 or downloaded >= content_length:
                elapsed = max(now - start_time, 0.001)
                speed_mb = (downloaded / (1024 * 1024)) / elapsed
                pct = (downloaded / content_length) * 100 if content_length > 0 else 0
                sys.stdout.write(
                    f"\r    Progress: {downloaded / (1024**2):.1f} / {content_length / (1024**2):.1f} MiB "
                    f"({pct:.1f}%) | Speed: {speed_mb:.1f} MiB/s"
                )
                sys.stdout.flush()
                last_print = now

    sys.stdout.write("\n")
    part_dest.replace(dest)

# Process all artifacts with strict idempotency and hash verification
for idx, item in enumerate(ARTIFACTS, 1):
    dest = item["dest"]
    expected_hash = item["sha256"]
    name = item["name"]
    print(f"\n[{idx}/{len(ARTIFACTS)}] Checking {name}...")

    # Step A: Check if existing file matches SHA-256
    if dest.exists():
        print(f"  File exists ({dest.stat().st_size / (1024**2):.1f} MiB). Verifying SHA-256...")
        actual_hash = compute_sha256(dest)
        if actual_hash == expected_hash:
            print(f"  [OK] Verified existing: {name} (SHA-256 match). Skipping download.")
            continue
        else:
            print(f"  [WARN] Hash mismatch on existing file (expected {expected_hash[:12]}..., got {actual_hash[:12]}...). Re-downloading.")
            dest.unlink()

    # Step B: Download directly from official Hugging Face resolve URL
    print(f"  Downloading from official source: {item['url']}")
    download_file(item["url"], dest, item["expected_bytes"])

    # Step C: Strict SHA-256 verification after download
    print(f"  Verifying downloaded file SHA-256...")
    verified_hash = compute_sha256(dest)
    if verified_hash != expected_hash:
        dest.unlink(missing_ok=True)
        raise RuntimeError(
            f"CRITICAL ABORT: SHA-256 verification failed for {name}!\n"
            f"  Expected: {expected_hash}\n"
            f"  Actual:   {verified_hash}\n"
            f"The corrupted download was removed. Please retry."
        )
    print(f"  [OK] SHA-256 Verified: {verified_hash[:16]}... Match confirmed!")

print("\n" + "=" * 70)
print("* ALL VTO MODEL WEIGHTS DOWNLOADED, VERIFIED, AND READY!")
print(f"  - MMDiT:     {ARTIFACTS[0]['dest']} (OK)")
print(f"  - YOLOX:     {ARTIFACTS[1]['dest']} (OK)")
print(f"  - DWPose:    {ARTIFACTS[2]['dest']} (OK)")
print("=" * 70)


In [ ]:
# 5. Connect ngrok Tunnel on Port 8001
from pyngrok import ngrok
import os

ngrok_token = os.environ.get('NGROK_AUTHTOKEN')
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)
else:
    print("NOTE: NGROK_AUTHTOKEN not set in Colab Secrets. Tunnel will run on standard unauthenticated session.")

ngrok.kill()  # Terminate previous sessions if any
tunnel = ngrok.connect(8001, 'http')

print('=' * 64)
print('* AURA VTO COLAB PUBLIC TUNNEL URL:')
print(f'  {tunnel.public_url}')
print('=' * 64)
print('Copy this URL into your AURA mobile app .env file:')
print(f'EXPO_PUBLIC_VTO_COLAB_URL={tunnel.public_url}')
print('=' * 64)


In [ ]:
# 6. Launch FastAPI VTO Inference Service
# Lifespan loads MMDiT + DWPose into GPU memory automatically.
# Once 'Lifespan startup completed' appears, the service is ready.
!uvicorn services.vto_gpu.app:app --host 0.0.0.0 --port 8001


In [ ]:
# 7. Optional Diagnostics & Sanity Check (Run in a parallel cell/session)
!curl -s http://127.0.0.1:8001/health
print()
!curl -s http://127.0.0.1:8001/ready
print()
!curl -s http://127.0.0.1:8001/v1/vto/diagnostics/auth
print()
